# Module 6 – Final AI & ML Project
## Student Performance Prediction using Machine Learning

**Project Type:** Machine Learning Classification

**Objective:** Build a complete beginner-friendly Machine Learning project that predicts whether a student is likely to **Pass** or **Need Support** based on study habits, attendance, previous performance, assignment completion, sleep, and participation.

### Project Workflow
1. Define the problem
2. Create and explore the dataset
3. Clean and prepare the data
4. Perform exploratory data analysis
5. Split the data into training and testing sets
6. Train Machine Learning models
7. Compare model performance
8. Evaluate the selected model
9. Understand feature importance
10. Test the model on new student data
11. Summarize findings and limitations

**Important:** This is an educational project using a simulated dataset. It should not be used to make real decisions about students.


## 1. Problem Statement

Students can have different study habits, attendance levels, assignment completion rates, and previous academic performance. The goal of this project is to demonstrate how Machine Learning can identify patterns in these variables and predict a simple outcome: **Pass** or **Need Support**.

The target is generated from a transparent educational rule with some random variation, so the notebook demonstrates a complete classification workflow without requiring an external data file.


## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

print("Libraries imported successfully!")


## 3. Create the Educational Dataset

The dataset is generated locally so the project is self-contained. A fixed random seed makes the experiment reproducible.

In [ ]:
np.random.seed(42)
n_students = 500

study_hours = np.clip(np.random.normal(5.5, 2.0, n_students), 1, 10)
attendance = np.clip(np.random.normal(82, 10, n_students), 50, 100)
previous_score = np.clip(np.random.normal(65, 15, n_students), 20, 100)
assignment_completion = np.clip(np.random.normal(78, 15, n_students), 20, 100)
sleep_hours = np.clip(np.random.normal(7, 1.2, n_students), 4, 10)
participation = np.clip(np.random.normal(6, 2.2, n_students), 1, 10)

performance_score = (
    5.0 * study_hours
    + 0.30 * attendance
    + 0.35 * previous_score
    + 0.20 * assignment_completion
    + 1.5 * sleep_hours
    + 2.0 * participation
    + np.random.normal(0, 7, n_students)
)

target = np.where(performance_score >= 105, "Pass", "Need Support")

df = pd.DataFrame({
    "Study_Hours_Per_Day": np.round(study_hours, 1),
    "Attendance_Percent": np.round(attendance, 1),
    "Previous_Score": np.round(previous_score, 1),
    "Assignment_Completion_Percent": np.round(assignment_completion, 1),
    "Sleep_Hours": np.round(sleep_hours, 1),
    "Participation_Score": np.round(participation, 1),
    "Performance": target
})

display(df.head())


## 4. Explore the Dataset

In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nPerformance distribution:")
print(df["Performance"].value_counts())
print("\nNumerical summary:")
display(df.describe())


## 5. Data Cleaning

Standard cleaning checks are demonstrated below.

In [ ]:
df = df.drop_duplicates().copy()

numeric_columns = df.select_dtypes(include=np.number).columns
df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].median())
df["Performance"] = df["Performance"].fillna(df["Performance"].mode()[0])

print("Rows after cleaning:", len(df))
print("Remaining missing values:", int(df.isnull().sum().sum()))


## 6. Exploratory Data Analysis – Performance Distribution

In [ ]:
df["Performance"].value_counts().plot(kind="bar", figsize=(7, 5))
plt.title("Student Performance Distribution")
plt.xlabel("Performance Class")
plt.ylabel("Number of Students")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Exploratory Data Analysis – Study Hours vs Previous Score

In [ ]:
for label in ["Pass", "Need Support"]:
    subset = df[df["Performance"] == label]
    plt.scatter(
        subset["Study_Hours_Per_Day"],
        subset["Previous_Score"],
        label=label,
        alpha=0.6
    )

plt.title("Study Hours vs Previous Score")
plt.xlabel("Study Hours Per Day")
plt.ylabel("Previous Score")
plt.legend()
plt.grid(True)
plt.show()


## 8. Prepare Features and Target

In [ ]:
feature_columns = [
    "Study_Hours_Per_Day",
    "Attendance_Percent",
    "Previous_Score",
    "Assignment_Completion_Percent",
    "Sleep_Hours",
    "Participation_Score"
]

X = df[feature_columns]
y = df["Performance"]

print("Features:", feature_columns)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


## 9. Split the Data

80% is used for training and 20% for testing. The test data remains unseen during model training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 10. Train Two Machine Learning Models

We compare:
- **Logistic Regression** – a simple, interpretable classification model.
- **Random Forest** – an ensemble model that can capture more complex relationships.


In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=6
)

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

print("Both models trained successfully!")


## 11. Compare Model Performance

In [ ]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model
}

comparison = []

for name, model in models.items():
    predictions = model.predict(X_test)
    comparison.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, pos_label="Pass"),
        "Recall": recall_score(y_test, predictions, pos_label="Pass"),
        "F1 Score": f1_score(y_test, predictions, pos_label="Pass")
    })

comparison_df = pd.DataFrame(comparison).sort_values("F1 Score", ascending=False)
display(comparison_df.style.format({
    "Accuracy": "{:.2%}",
    "Precision": "{:.2%}",
    "Recall": "{:.2%}",
    "F1 Score": "{:.2%}"
}))


## 12. Select and Evaluate the Final Model

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = models[best_model_name]
y_pred = best_model.predict(X_test)

print("Selected model:", best_model_name)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


## 13. Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title(f"Confusion Matrix – {best_model_name}")
plt.show()


## 14. Feature Importance

In [ ]:
if best_model_name == "Random Forest":
    importance_values = best_model.feature_importances_
else:
    importance_values = np.abs(best_model.named_steps["model"].coef_[0])

importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": importance_values
}).sort_values("Importance", ascending=False)

display(importance_df)

importance_df.sort_values("Importance").plot(
    x="Feature", y="Importance",
    kind="barh", figsize=(8, 5), legend=False
)
plt.title(f"Feature Importance – {best_model_name}")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 15. Test the Model on New Student Data

In [ ]:
new_students = pd.DataFrame({
    "Study_Hours_Per_Day": [8.0, 4.0, 6.0],
    "Attendance_Percent": [95.0, 65.0, 82.0],
    "Previous_Score": [82.0, 48.0, 67.0],
    "Assignment_Completion_Percent": [92.0, 55.0, 80.0],
    "Sleep_Hours": [7.5, 5.5, 7.0],
    "Participation_Score": [9.0, 3.0, 6.0]
})

new_predictions = best_model.predict(new_students)
new_students["Predicted_Performance"] = new_predictions

display(new_students)


## 16. Understand the Results

The model learns statistical patterns from the simulated training data. Variables such as previous score, attendance, study time, and assignment completion can contribute to predictions.

**Important:** Feature importance does not prove that a variable causes a student to pass or need support. It only describes how the trained model used the variables in this dataset.


## 17. Limitations and Ethical Considerations

- The dataset is simulated for educational purposes and is not a real student dataset.
- Results depend on the data and assumptions used to create the dataset.
- A real educational system would require a larger, representative, carefully collected dataset.
- Predictions should not be the sole basis for decisions about students.
- Student information can be sensitive and should be handled with appropriate privacy and security protections.
- Real-world deployment would require validation, monitoring, fairness checks, and appropriate human oversight.


## 18. Final Conclusion

This project demonstrates a complete Machine Learning workflow from problem definition to prediction.

### What I learned
- Define a classification problem.
- Prepare and explore a dataset.
- Split data into training and testing sets.
- Train Logistic Regression and Random Forest models.
- Compare models using accuracy, precision, recall, and F1 score.
- Use a confusion matrix to understand predictions.
- Inspect feature importance.
- Make predictions on new data.
- Consider limitations, privacy, and responsible AI.

**Final takeaway:** A successful ML project is not only about training a model; it also requires understanding the data, evaluating the model, communicating results, and considering responsible use.


## 19. GitHub Repository Information

### Suggested repository name
`module6-student-performance-ml-project`

### Suggested repository description
`Final AI & ML project using Python and Scikit-learn to demonstrate student performance classification, model comparison, evaluation, and prediction.`

### Suggested README content

```text
# Student Performance Prediction – Machine Learning Project

## Overview
A beginner-friendly final Machine Learning project that predicts student performance using study habits, attendance, previous scores, assignment completion, sleep, and participation.

## Technologies
- Python
- Pandas
- NumPy
- Matplotlib
- Scikit-learn
- Google Colab

## Models
- Logistic Regression
- Random Forest Classifier

## Project Workflow
Data creation → exploration → cleaning → visualization → train/test split → model training → model comparison → evaluation → feature importance → new predictions.

## Important Note
The dataset is simulated for educational purposes. The model should not be used to make real decisions about students.
```


## 20. Suggested LinkedIn Post

🚀 **Completed my Final AI & ML Project!**

I built a **Student Performance Prediction** Machine Learning project using Python and Scikit-learn.

In this project, I worked through a complete ML workflow:

🔹 Data preparation and exploration  
🔹 Exploratory data visualization  
🔹 Train/test split  
🔹 Logistic Regression and Random Forest models  
🔹 Model comparison using accuracy, precision, recall, and F1 score  
🔹 Confusion matrix evaluation  
🔹 Feature importance analysis  
🔹 Predictions on new student data

I also learned why responsible AI and data privacy are important when working with educational data.

🔗 GitHub Project: [PASTE YOUR GITHUB LINK HERE]

#AI #MachineLearning #Python #ScikitLearn #DataScience #GoogleColab #StudentProject #Learning
